# GigaAM: микрофон → текст, VS Code + Python 3.13

**Отдельный notebook для уже установленного обычного 64-bit Python 3.13, включая 3.13.11.** Один локальный kernel: второе ASR-окружение, Python 3.12 и сервер не нужны. Git не нужен на рабочем компьютере при установке готовых wheels; исходный requirements с Git URL требует Git на разрешённой машине подготовки. Модель прежняя: GigaAM Multilingual 220M CTC, RU / KK / EN, автоматический смешанный язык. Start записывает, Stop выключает микрофон и распознаёт фрагмент. Это не непрерывные субтитры.

## Что подготовить один раз
1. Скачайте ZIP этого репозитория, распакуйте и откройте папку в VS Code. Откройте этот notebook; справа сверху **Select Kernel → ваш локальный Python 3.13.11**. Нужны расширения Python/Jupyter и `ipykernel` в этом же Python. Не выбирайте WSL, SSH, Colab или JupyterLite.
2. Зависимости: передайте `requirements-microphone-py313.txt` вашему разрешённому установщику. Если прямой интернет запрещён, используйте корпоративный процесс: готовые пакеты или согласованную папку wheels. **Notebook ничего не скачивает автоматически и не меняет настройки сети.** Подробности для администратора: [PYTHON313_VSCODE.md](../docs/PYTHON313_VSCODE.md). Если пакеты уже установлены, пропустите установку.
3. Нужны локальный FFmpeg и два проверенных файла модели: `config.json`, `pytorch_model.bin` (~883 MB). Положите их в `models/gigaam` внутри проекта или задайте свой путь ниже. Модель получается отдельно через разрешённый процесс; наличие Python-пакетов не означает наличие весов.
4. Выполните ячейки сверху вниз, нажмите Start, произнесите фразу, нажмите Stop. Разрешение микрофона для настольных приложений должно быть включено в Windows.

Записывайте только с согласия участников. Аудио обрабатывается локально, временные WAV удаляются после обработки. На одну запись: максимум 30 секунд / 5 MB. Текст может сохраняться в outputs/state notebook: очищайте их перед пересылкой. Аварийное завершение kernel может оставить временный каталог.

In [ ]:
# 1. НАСТРОЙКИ: обычно нужно изменить только MODEL или FFMPEG, если они не в проекте/PATH.
from pathlib import Path
import os, sys, platform, shutil, subprocess

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "app/asr.py").is_file()), None)
# Если папка проекта не найдена, раскомментируйте и укажите вашу:
# REPO = Path(r"C:\Users\you\Downloads\teams-local-transcriber-main")
MODEL = REPO / "models/gigaam" if REPO else Path("models/gigaam")
# MODEL = Path(r"D:\approved-models\gigaam")
FFMPEG = shutil.which("ffmpeg")  # либо r"C:\approved-tools\ffmpeg\bin\ffmpeg.exe"
INPUT_DEVICE = None  # системный микрофон; ID можно выбрать ниже
INFERENCE_DEVICE = "cpu"  # этот простой вариант проверяется на CPU
ASR_PY = Path(sys.executable)  # ровно тот Python, который выбран в VS Code

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f"Выберите kernel Python 3.13. Сейчас: {sys.version.split()[0]} ({sys.executable})")
if platform.architecture()[0] != "64bit":
    raise RuntimeError("Нужен 64-bit Python 3.13")
import sysconfig
if sysconfig.get_config_var("Py_GIL_DISABLED"):
    raise RuntimeError("Нужен обычный CPython 3.13, не free-threaded 3.13t")
print("Python:", sys.version.split()[0], "|", sys.executable)
print("Проект:", REPO, "| Модель:", MODEL)

### Только если у вас уже есть согласованная папка wheels

По умолчанию следующая ячейка **ничего не устанавливает**. Если ИТ подготовили полный набор для **Windows x64 / CPython 3.13**, укажите папку и установите `INSTALL_FROM_WHEELHOUSE = True`. В наборе должен быть wheel GigaAM, собранный из закреплённого исходного commit (см. инструкцию). Интернет при этой установке запрещён параметром `--no-index`; пакеты из исходников не собираются. После установки перезапустите kernel и снова выполните ячейки, вернув флаг в False. Не переносите Linux wheels на Windows.

In [ ]:
INSTALL_FROM_WHEELHOUSE = False
WHEELHOUSE = Path(r"C:\approved\gigaam-wheels")
if INSTALL_FROM_WHEELHOUSE:
    if REPO is None or not WHEELHOUSE.is_dir():
        raise RuntimeError("Укажите папку проекта REPO и существующую папку WHEELHOUSE")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",
                    "--find-links", str(WHEELHOUSE), "-r",
                    str(REPO / "requirements-microphone-py313-offline.txt")],
                   check=True, timeout=1800)
    print("Установлено. Перезапустите kernel; верните INSTALL_FROM_WHEELHOUSE = False.")
else:
    print("Установка пропущена. Используем пакеты текущего kernel.")

In [ ]:
# 2. ПРОВЕРКА. Не открывает микрофон, не загружает веса в RAM и не обращается к сети.
if REPO is None or not (REPO / "app/asr.py").is_file():
    raise RuntimeError("Распакуйте ZIP проекта и задайте REPO в первой ячейке")
if not FFMPEG or not Path(FFMPEG).is_file():
    raise RuntimeError("Укажите полный путь к согласованному ffmpeg.exe в первой ячейке")
try:
    import sounddevice as sd
    import ipywidgets as widgets
    from IPython.display import display
except OSError as exc:
    raise RuntimeError("Не загружается PortAudio: используйте Windows wheel sounddevice; на Linux нужна системная libportaudio2. Затем Restart Kernel.") from exc
except ImportError as exc:
    raise RuntimeError("В выбранном kernel нужны sounddevice, ipywidgets и ipykernel из requirements-microphone-py313.txt; затем Restart Kernel") from exc
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from app.asr import verify_gigaam_files
if not all((MODEL / name).is_file() for name in ("config.json", "pytorch_model.bin")):
    raise RuntimeError(f"Нет файлов модели в {MODEL}. Укажите MODEL или получите оба файла через согласованный процесс.")
verify_gigaam_files(MODEL)  # SHA256; проверка 883 MB может занять время
probe = subprocess.run([sys.executable, "-c",
    "import torch, torchaudio, numpy, onnx, onnxruntime; from gigaam.model import GigaAMASR; "
    "assert torch.__version__.split('+')[0]=='2.10.0'; "
    "assert torchaudio.__version__.split('+')[0]=='2.10.0'; print('ASR imports OK')"],
    cwd=REPO, capture_output=True, text=True, timeout=120,
    env=dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUTF8="1"))
if probe.returncode:
    raise RuntimeError("ASR-пакеты отсутствуют/несовместимы. Установите закреплённые requirements в этот же Python; ошибка импорта доступна локально в probe.stderr.")
subprocess.run([str(FFMPEG), "-version"], check=True, stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL, timeout=10)
print("Проверки пройдены. Модель локальная; kernel:", sys.executable)
print(sd.query_devices())  # только список, запись ещё не началась

## Start / Stop

При необходимости измените `INPUT_DEVICE` в настройках на числовой ID из списка. Start открывает микрофон; Stop останавливает запись и запускает локальный ASR. Новая запись доступна после окончания обработки. «Закрыть / отменить» останавливает запись/ASR. На медленном CPU первый результат может появляться с задержкой: модель загружается заново для каждого фрагмента. Веса не уходят в сеть.

In [ ]:
# MIC_CONTROLLER: self-contained; no notebook helper download needed.
import atexit, base64, json, signal, tempfile, threading, time, wave

MAX_SECONDS = 30
MAX_BYTES = 5_000_000
INFERENCE_TIMEOUT = 600
WORKER_CODE = r"""
import json, sys
from pathlib import Path
from app.asr import ASRSettings, LocalASR
settings = ASRSettings(backend="gigaam", model_path=Path(sys.argv[2]),
    device=sys.argv[3], max_audio_seconds=30, max_input_bytes=5_000_000,
    ffmpeg_binary=sys.argv[4], ffmpeg_timeout_seconds=60, temp_root=Path(sys.argv[1]).parent)
segments = LocalASR(settings).transcribe(Path(sys.argv[1]))
print(json.dumps({"text": " ".join(s["text"] for s in segments)}, ensure_ascii=False))
"""

class MicrophoneNotebook:
    def __init__(self):
        self.stop_event = threading.Event()
        self.cancel_event = threading.Event()
        self.thread = None
        self.closed = False
        self.start_button = widgets.Button(description="Start", button_style="success")
        self.stop_button = widgets.Button(description="Stop", button_style="warning", disabled=True)
        self.close_button = widgets.Button(description="Закрыть / отменить")
        self.download_button = widgets.Button(description="Подготовить TXT", disabled=True)
        self.status = widgets.Label(value="Готово к записи")
        self.result = widgets.Textarea(value="", description="Текст:", layout=widgets.Layout(width="100%", height="180px"))
        self.download = widgets.HTML(value="")
        self.start_button.on_click(lambda _: self.start())
        self.stop_button.on_click(lambda _: self.stop_event.set())
        self.close_button.on_click(lambda _: self.close())
        self.download_button.on_click(lambda _: self.prepare_download())
        self.ui = widgets.VBox([widgets.HBox([self.start_button, self.stop_button, self.close_button]), self.status, self.result, self.download_button, self.download])
        atexit.register(self.close)

    def start(self):
        if self.closed or (self.thread and self.thread.is_alive()):
            return
        self.stop_event.clear()
        self.cancel_event.clear()
        self.result.value = ""
        self.download.value = ""
        self.download_button.disabled = True
        self.start_button.disabled = True
        self.stop_button.disabled = False
        self.thread = threading.Thread(target=self._record_and_transcribe, daemon=True)
        self.thread.start()

    def _record_and_transcribe(self):
        pcm = bytearray()
        callback_error = []
        try:
            device = sd.query_devices(INPUT_DEVICE, "input")
            rate = int(round(float(device["default_samplerate"])))
            if not 8000 <= rate <= 384000:
                raise ValueError("Неожиданная частота микрофона")
            sd.check_input_settings(device=INPUT_DEVICE, channels=1, dtype="int16", samplerate=rate)
            # Leave room for WAV header; keep full int16 samples only.
            cap = min(rate * MAX_SECONDS * 2, MAX_BYTES - 4096)
            cap -= cap % 2
            deadline = time.monotonic() + MAX_SECONDS
            def callback(indata, frames, timing, status):
                if self.stop_event.is_set() or self.cancel_event.is_set():
                    raise sd.CallbackStop
                try:
                    if status:
                        callback_error.append("Сбой/переполнение аудиовхода; повторите запись")
                        self.stop_event.set()
                        raise sd.CallbackStop
                    remaining = cap - len(pcm)
                    pcm.extend(bytes(indata)[:remaining])
                    if len(pcm) >= cap or time.monotonic() >= deadline:
                        self.stop_event.set()
                        raise sd.CallbackStop
                except sd.CallbackStop:
                    raise
                except Exception:
                    callback_error.append("Ошибка аудиозаписи")
                    self.stop_event.set()
                    raise sd.CallbackAbort
            self.status.value = "Запись… нажмите Stop (максимум 30 секунд / 5 MB)"
            with sd.RawInputStream(device=INPUT_DEVICE, samplerate=rate, channels=1,
                    dtype="int16", blocksize=1024, callback=callback,
                    finished_callback=self.stop_event.set):
                self.stop_event.wait(MAX_SECONDS)
                self.stop_event.set()
            self.stop_button.disabled = True
            if self.cancel_event.is_set():
                return
            if callback_error:
                raise RuntimeError(callback_error[0])
            if len(pcm) < rate // 5 * 2:
                raise ValueError("Запись слишком короткая: произнесите фразу дольше 0.2 секунды")
            self.status.value = "Микрофон выключен. Локальное распознавание…"
            with tempfile.TemporaryDirectory(prefix="gigaam-microphone-") as temp:
                wav_path = Path(temp) / "microphone.wav"
                with wave.open(str(wav_path), "wb") as audio:
                    audio.setnchannels(1)
                    audio.setsampwidth(2)
                    audio.setframerate(rate)
                    audio.writeframes(pcm)
                pcm.clear()
                if wav_path.stat().st_size > MAX_BYTES:
                    raise ValueError("Превышен лимит WAV")
                env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUTF8="1")
                command = [str(ASR_PY), "-c", WORKER_CODE, str(wav_path), str(MODEL), INFERENCE_DEVICE, str(FFMPEG)]
                with subprocess.Popen(command, cwd=REPO, env=env, stdin=subprocess.DEVNULL,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding="utf-8",
                        start_new_session=(os.name != "nt"),
                        creationflags=(subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)) as process:
                    started = time.monotonic()
                    try:
                        while True:
                            if self.cancel_event.is_set():
                                raise InterruptedError("Отменено")
                            if time.monotonic() - started > INFERENCE_TIMEOUT:
                                raise TimeoutError("Лимит распознавания 600 секунд; попробуйте короче или более мощную машину")
                            try:
                                stdout, stderr = process.communicate(timeout=0.2)
                                break
                            except subprocess.TimeoutExpired:
                                continue
                        if process.returncode:
                            # No audio/text/traceback is printed automatically.
                            raise RuntimeError("ASR завершился с ошибкой: проверьте зависимости, модель, FFmpeg и RAM")
                        payload = json.loads(stdout.strip().splitlines()[-1])
                        self.result.value = payload["text"]
                        self.download_button.disabled = not bool(self.result.value)
                        self.status.value = "Готово" if self.result.value else "Текст не получен; попробуйте другую фразу"
                    finally:
                        if process.poll() is None:
                            self._kill_process_tree(process)
                        process.communicate()
        except InterruptedError:
            self.status.value = "Отменено"
        except Exception as error:
            self.status.value = f"Ошибка: {error}"
        finally:
            pcm.clear()
            self.stop_event.set()
            self.stop_button.disabled = True
            self.start_button.disabled = self.closed
            if self.closed:
                self.status.value = "Закрыто. Для новой записи выполните эту ячейку заново."

    @staticmethod
    def _kill_process_tree(process):
        if os.name == "nt":
            # FFmpeg is a child too; terminate the tree before deleting its WAVs.
            try:
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=10, check=False)
            finally:
                if process.poll() is None:
                    process.kill()
        else:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    def prepare_download(self):
        if not self.result.value:
            return
        encoded = base64.b64encode(self.result.value.encode("utf-8")).decode("ascii")
        self.download.value = '<a download="gigaam-transcript.txt" href="data:text/plain;charset=utf-8;base64,' + encoded + '">Скачать TXT</a>'
        # Some VS Code renderers block data links; copying text remains available.

    def close(self):
        self.closed = True
        self.cancel_event.set()
        self.stop_event.set()
        if self.thread and self.thread is not threading.current_thread():
            self.thread.join(timeout=5)
        self.start_button.disabled = True
        self.stop_button.disabled = True
        self.status.value = "Закрыто" if not self.thread or not self.thread.is_alive() else "Остановка…"
        atexit.unregister(self.close)

if "microphone" in globals():
    microphone.close()
    if microphone.thread and microphone.thread.is_alive():
        raise RuntimeError("Предыдущая запись ещё закрывается; подождите и повторите ячейку")
microphone = MicrophoneNotebook()
display(microphone.ui)

## Быстрый тест и устранение ошибок
- RU: «Сегодня обсудим план проекта и следующую встречу».
- KK: «Бүгін біз жобаның жоспарын талқылаймыз».
- EN: “Today we will review the project plan.”
- Смешанная фраза: «Бүгін meeting agenda бойынша сөйлесеміз, затем проверим сроки».

Это тестовые фразы, не гарантии точности. Английский и переключение языков требуют собственной проверки. Модель не выполняет диаризацию. «Подготовить TXT» создаёт ссылку только по клику; если VS Code её блокирует, скопируйте текст из поля.

- `ModuleNotFoundError`: пакеты должны быть в Python, путь к которому печатает первая ячейка; не в другом kernel. После установки Restart Kernel.
- `No matching distribution`: набор должен быть для обычного CPython 3.13 / Windows x64. 3.13t, Windows ARM64 и 32-bit не заявлены как поддерживаемые этим набором.
- Сетевые ошибки установки: обратитесь к разрешённому корпоративному установщику/зеркалу. Не отключайте TLS и не меняйте proxy/firewall ради notebook.
- `PortAudio` / microphone error: проверьте настройки Windows → Privacy & security → Microphone → доступ настольным приложениям, ID устройства и что kernel локальный. При отсутствии mono-входа выберите другой микрофон.
- Ошибка ASR: проверьте FFmpeg, модель и свободную RAM (ориентир 4–8 GB). Максимум ожидания 600 секунд; начните с короткой фразы.

Границы проверки и точные результаты: [PYTHON313_VSCODE.md](../docs/PYTHON313_VSCODE.md). Реальные Windows-микрофон, VS Code rendering и корпоративная установка требуют проверки на вашем устройстве. Notebook не изменяет разрешения ОС автоматически.